In [ ]:
!pip install -q numpy scikit-learn sentence-transformers

In [ ]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")
    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")
    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]
def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing lab6_search.py


In [1]:
%%writefile task1_tokens.py
import re
samples = ["AI, search!", "I can't log in.", "help@example.com",
           "یہ ایک مثال ہے", "", "CS4944 version 2.1"]
for text in samples:
    ws = text.split()
    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    print(repr(text), "whitespace:", ws, len(ws))
    print("word-like:", words, len(words))
assert re.findall(r"\b\w+\b", "") == []
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
for text in samples[:2]:
    print(text, model.tokenizer.tokenize(text))

Writing task1_tokens.py


In [ ]:
%%writefile guided_chatbot.py
from lab6_search import Retriever
FAQ = [
    {"id": "faq:rank", "question": "Can Superior rank?",
     "answer": "Judging by Superior's behavior, never.",
     "source": "campus_notes.txt"},
    {"id": "faq:road",
     "question": "Is there a road in Superior, or is Superior itself on the road?",
     "answer": "Brother, Superior itself is on the road.",
     "source": "campus_notes.txt"},
    {"id": "faq:admission",
     "question": "Is Superior famous enough to take admission there?",
     "answer": "Oh brother, it is not famous at all. Take admission somewhere else.",
     "source": "campus_notes.txt"},
    {"id": "faq:timetable_app",
     "question": "Why didn't Superior build Sir Rasikh's timetable app?",
     "answer": "Superior did not have that much budget, and Sir demanded a lot.",
     "source": "campus_notes.txt"},
    {"id": "faq:world_map",
     "question": "Will Sir Rasikh build the full 3D world map solution?",
     "answer": "Why not, Sir Rasikh will do it. He is just not in form right now.",
     "source": "campus_notes.txt"}]
def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)
def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"],
                        default="tfidf")
    args = parser.parse_args()
    engine = build_bot(args.backend)
    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            query = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        print("Bot:", reply(engine, query))

Overwriting guided_chatbot.py


In [ ]:
!python task1_tokens.py

'AI, search!' whitespace: ['AI,', 'search!'] 2
word-like: ['AI', 'search'] 2
"I can't log in." whitespace: ['I', "can't", 'log', 'in.'] 4
word-like: ['I', 'can', 't', 'log', 'in'] 5
'help@example.com' whitespace: ['help@example.com'] 1
word-like: ['help', 'example', 'com'] 3
'یہ ایک مثال ہے' whitespace: ['یہ', 'ایک', 'مثال', 'ہے'] 4
word-like: ['یہ', 'ایک', 'مثال', 'ہے'] 4
'' whitespace: [] 0
word-like: [] 0
'CS4944 version 2.1' whitespace: ['CS4944', 'version', '2.1'] 3
word-like: ['CS4944', 'version', '2', '1'] 4
modules.json: 100% 349/349 [00:00<00:00, 893kB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 523kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 22.6MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 238kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.84MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:02<00:00, 68.0MB/s, 7.42MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:02<00:00, 32.4MB/s, 7.75MB/s  ]
model.sa

In [ ]:
%run guided_chatbot.py

FAQ assistant. Type exit to stop.
You: can supeior rank
Bot: Possible matches; choose or rephrase your question:
Can Superior rank? (similarity 0.525)
Is there a road in Superior, or is Superior itself on the road? (similarity 0.000)
Is Superior famous enough to take admission there? (similarity 0.000)
You: Is Superior famous enough to take admission there?
Bot: Oh brother, it is not famous at all. Take admission somewhere else. [source: campus_notes.txt]
